# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR⁲ Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL.


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
# Print dataset description and title
print(f"{dataset.metadata.name}: {dataset.metadata.description}")


## 2. Data Overview
Review available record sets and their fields, referencing by `@id`. This step helps to understand the dataset's structure for further exploration.


In [ ]:
# List all available record sets and their fields by @id
record_sets = list(dataset.metadata.record_sets)
print(f"Found {len(record_sets)} record set(s):\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    field_ids = [f.id for f in rs.fields]
    field_names = [f.name for f in rs.fields]
    print(f"  Field @ids: {field_ids}")
    print(f"  Field names: {field_names}")
    print("")
# For demonstration, show a couple of records from the first RecordSet
if record_sets:
    sample_rs_id = record_sets[0].id
    print(f"Sample records for RecordSet @id={sample_rs_id}:")
    for i, rec in enumerate(dataset.records(record_set=sample_rs_id)):
        print(rec)
        if i == 2:
            break


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All references to record set and fields are by their `@id`.

In [ ]:
# Extract all data into DataFrames, keyed by record set @id
rs_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"Available record set @ids: {list(dataframes.keys())}")
# Display columns of the first record set DataFrame
if rs_ids:
    main_rs = rs_ids[0]
    print(f"Columns for record set '@id': {main_rs}")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())


## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group by fields, using `@id` references for fields.

In [ ]:
# Example: Use a numeric field and a group field, referencing @id
main_df = dataframes[main_rs]
# Find first numeric field
record_set = next(rs for rs in dataset.metadata.record_sets if rs.id == main_rs)
numeric_field = None
group_field = None
for field in record_set.fields:
    # Field may specify type as e.g., 'Float', 'Integer', etc.
    if hasattr(field, 'data_type') and str(field.data_type).lower() in ('float', 'integer', 'number'):
        numeric_field = field.id
        break
# Choose the next textual field as group_field if available
for field in record_set.fields:
    if hasattr(field, 'data_type') and str(field.data_type).lower() in ('text', 'string'):
        group_field = field.id
        break

if numeric_field and numeric_field in main_df.columns:
    # Remove NaNs and outliers for demonstration
    df_num = main_df[[numeric_field]].dropna()
    threshold = df_num[numeric_field].mean() + 2 * df_num[numeric_field].std()
    filtered_df = main_df[main_df[numeric_field] < threshold]
    print(f"Filtered records with {numeric_field} < {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize
    col_z = f"{numeric_field}_normalized"
    filtered_df[col_z] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, col_z]].head())
else:
    print("No suitable numeric field found for EDA.")

# Group by group_field if present
if group_field and group_field in main_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable group field found for grouping.")


## 5. Visualization
Visualize data distributions and relationships between fields. This uses the normalized numeric field and the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field and numeric_field in main_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

# Boxplot by group field
if group_field and numeric_field and group_field in main_df.columns and numeric_field in main_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
    plt.title(f"{numeric_field} grouped by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Insufficient fields for group-wise visualization.")


## 6. Conclusion
In this notebook, we've loaded the FAIR⁲ Croissant dataset for Mass Spectrometry Analysis of Extracellular Vesicles from Stimulated Human Mast Cells using only `@id` references for all entities. We explored the dataset structure, extracted data, performed basic EDA, normalized and grouped data, and visualized field distributions. For deeper insights, you may repeat analyses for other record sets or fields as required.